# G12: G9's eval_mini on OLMoE GGUFs (Colab T4)

Runs G9's **mini** profile (ARC-C 500, HellaSwag 400, MMLU 570, GSM8K 200, TruthfulQA-mc2 200; same shots, limits
and seed as G9) on **bf16 and six llama.cpp formats**, all through the same backend (llama-cpp-python, in-process;
see `llamacpp_lm.py` for why not llama-server), with **per-item logs**, then compares each format with bf16 item by
item (`paired.py`).

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

- Results go to Google Drive (`DRIVE_OUT` below) after every task. If the session disconnects, run all again:
  finished tasks are skipped and the llama-cpp-python build is reused from Drive.
- Time (estimate): first session +15-25 min to compile llama-cpp-python and +10 min to download and convert the
  model; then ~25 min per quantized format and ~1-1.5 h for bf16.
- bf16 (13.8 GB) may not fit the T4's 15 GB with everything on the GPU; the run then retries with
  `BF16_FALLBACK_GPU_LAYERS` layers on the GPU (slower) and records what it used.

In [ ]:
# --- settings -----------------------------------------------------------------------------------------------
FORMATS   = ["bf16", "Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q3_K_M", "Q2_K"]   # bf16 first: it is the baseline
MODEL, REVISION = "allenai/OLMoE-1B-7B-0924", "6d84c48581ece794365f2b8e9cfb043c68ade9c5"  # same pin as the team runs
LLAMA_TAG = "b11669"                       # llama.cpp build used for conversion/quantization in the team runs
LLAMA_CPP_PYTHON = "0.3.36"                # checked on the Mac: correct log-probabilities in-process
TEAM_RAW = "https://raw.githubusercontent.com/SreekanthZipsy/era-v5-lab/shubham_eval_mini/teams/12-quantization/eval_mini"
G9_RAW   = "https://raw.githubusercontent.com/theschoolofai/era-v5-lab/main/teams/09-evaluation/harness/run_eval_mini.py"
DRIVE_OUT = "/content/drive/MyDrive/era-v5-g12/eval_mini"   # results + the compiled wheel live here
WORK      = "/content/work"                                 # model files (local disk, rebuilt per session)
BF16_FALLBACK_GPU_LAYERS = 12              # of 16; used only if bf16 does not fit fully on the GPU
LIMIT = None                               # e.g. 5 for a quick end-to-end test; None = G9's full mini profile

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, json, glob
os.makedirs(DRIVE_OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
# llama-cpp-python with CUDA: compiled once (~15-25 min on Colab's 2 CPUs), then reused from Drive as a wheel
wheels = glob.glob(f"{DRIVE_OUT}/wheels/llama_cpp_python-{LLAMA_CPP_PYTHON}-*.whl")
if not wheels:
    os.makedirs(f"{DRIVE_OUT}/wheels", exist_ok=True)
    !CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75" pip wheel --no-deps --no-cache-dir -w {DRIVE_OUT}/wheels llama-cpp-python=={LLAMA_CPP_PYTHON}
    wheels = glob.glob(f"{DRIVE_OUT}/wheels/llama_cpp_python-{LLAMA_CPP_PYTHON}-*.whl")
!pip install -q {wheels[0]} lm_eval==0.4.13 datasets==5.1.0
import llama_cpp
assert llama_cpp.llama_supports_gpu_offload(), "llama-cpp-python was built without CUDA"
print("llama-cpp-python", llama_cpp.__version__, "with GPU offload")

In [ ]:
# code: our three files from the team repo, G9's runner (profile + record format) from upstream main
CODE = f"{WORK}/eval_mini"; os.makedirs(CODE, exist_ok=True)
for f in ["llamacpp_lm.py", "run_mini_gguf.py", "paired.py"]:
    !curl -fsSL {TEAM_RAW}/{f} -o {CODE}/{f}
!curl -fsSL {G9_RAW} -o {CODE}/run_eval_mini.py
!ls -la {CODE}

In [ ]:
# bf16 GGUF: llama.cpp converter at the team's build, pinned checkpoint; llama-quantize built CPU-only (~5 min)
NAME = MODEL.split("/")[-1]
BF16 = f"{WORK}/{NAME}-bf16.gguf"
if not os.path.exists(f"{WORK}/llama.cpp"):
    !git clone -q --depth 1 --branch {LLAMA_TAG} https://github.com/ggml-org/llama.cpp {WORK}/llama.cpp
    !pip install -q {WORK}/llama.cpp/gguf-py sentencepiece
    !cmake -S {WORK}/llama.cpp -B {WORK}/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF > /dev/null
    !cmake --build {WORK}/llama.cpp/build -j --target llama-quantize > /dev/null
if not os.path.exists(BF16):
    from huggingface_hub import snapshot_download
    hf = snapshot_download(MODEL, revision=REVISION, local_dir=f"{WORK}/hf",
                           allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"])
    !python {WORK}/llama.cpp/convert_hf_to_gguf.py {hf} --outtype bf16 --outfile {BF16}
    !rm -rf {WORK}/hf
!ls -la {WORK}/*.gguf

In [ ]:
# run every format; each task is saved to Drive as soon as it finishes, finished tasks are skipped
G9_TASKS = ["arc_challenge", "hellaswag", "mmlu", "gsm8k", "truthfulqa_mc2"]

def finished(fmt):
    p = f"{DRIVE_OUT}/runs/{fmt}/summary.json"
    return os.path.exists(p) and {r["task"] for r in json.load(open(p))["records"]} >= set(G9_TASKS)

def run(fmt, gguf, gpu_layers):
    cmd = (f"cd {CODE} && python run_mini_gguf.py --gguf {gguf} --format {fmt} --out {DRIVE_OUT}/runs "
           f"--g9-harness {CODE}/run_eval_mini.py --n-gpu-layers {gpu_layers}" + (f" --limit {LIMIT}" if LIMIT else ""))
    return subprocess.run(cmd, shell=True).returncode

for fmt in FORMATS:
    if finished(fmt):
        print(f"{fmt}: done, skipped"); continue
    gguf = BF16 if fmt == "bf16" else f"{WORK}/{NAME}-{fmt}.gguf"
    if not os.path.exists(gguf):   # same rule as scripts/quantize_eval.py: output layer Q8_0 below 8 bits
        extra = "" if fmt == "Q8_0" else "--output-tensor-type q8_0"
        !{WORK}/llama.cpp/build/bin/llama-quantize {extra} {BF16} {gguf} {fmt} > {WORK}/quantize-{fmt}.log 2>&1
    rc = run(fmt, gguf, -1)
    if rc != 0 and fmt == "bf16":
        print(f"bf16 failed with all layers on the GPU (exit {rc}); retrying with {BF16_FALLBACK_GPU_LAYERS} GPU layers")
        rc = run(fmt, gguf, BF16_FALLBACK_GPU_LAYERS)
    if rc != 0:
        raise SystemExit(f"{fmt} failed (exit {rc}); rerun this cell to resume")
    if fmt != "bf16":
        os.remove(gguf)            # keep the local disk small; bf16 stays until all formats are done

In [ ]:
# results: scores per format (G9's primary metrics) and the paired comparison with bf16, both saved to Drive
import pandas as pd
rows = []
for p in sorted(glob.glob(f"{DRIVE_OUT}/runs/*/summary.json")):
    s = json.load(open(p))
    for r in s["records"]:
        if r["primary"]:
            rows.append({"format": s["run_meta"]["format"], "task": r["task"], "value": round(100 * r["value"], 1),
                         "stderr": round(100 * (r["stderr"] or 0), 1), "n": r["n_items"], "seconds": r["wall_clock_s"],
                         "gpu_layers": s["run_meta"]["n_gpu_layers"]})
scores = pd.DataFrame(rows).pivot_table(index="format", columns="task", values="value").reindex(FORMATS)
scores.to_csv(f"{DRIVE_OUT}/scores.csv"); display(scores)
!python {CODE}/paired.py {DRIVE_OUT}/runs --base bf16 --out {DRIVE_OUT}/paired.md

## Bring the results back

Download `DRIVE_OUT` (the `runs/` folder with `summary.json` + `*.samples.jsonl` per format, `scores.csv`,
`paired.md`) and put it under `teams/12-quantization/eval_mini/results/` in the repo.